# Notebook Requirements
Run this code to execute the notebook if you didn't already cloned the repo.

In [ ]:
!git clone https://github.com/GiuseppeDaddario/Computer-Vision.git --recurse-submodules
%cd Computer-Vision

# Imports

#### Yolo + Baseline

In [ ]:
%pip install ultralytics --quiet
%pip install -U gdown

In [ ]:
# -------- Standard Library --------- #
import os
os.environ['WANDB_MODE'] = 'disabled'
import io
import random
import shutil
import multiprocessing
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor
import time

# -------- Other Libraries --------- #
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import cv2
from PIL import Image
from tqdm import tqdm
from ultralytics import YOLO

# -------- PyTorch --------- #
import torch
import torch.nn as nn
import torch.optim as optim
from torch import autocast
from torch.amp import GradScaler
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.models as models

# -------- Torchvision --------- #
import torchvision.transforms as T
import torchvision.transforms.functional as TF
from torchvision import transforms, models, datasets

# -------- Local Imports --------- #
from src import YOLOv5_training, YOLOv5_inference

# Globals

In [ ]:
# -------- General paths --------- #
DATASET_PATH = "dataset/CCPD2019"
TRAINING_PATH = "dataset/CCPD2019/ccpd_base"
TEST_DIR = "ccpd_challenge"
TEST_PATH = f"dataset/CCPD_YOLO/{TEST_DIR}/images/test" 

# -------- YOLO paths --------- #
DATASET_PATH_YOLO = "dataset/CCPD_YOLO"
TRAINING_CONFIG_YOLO = "dataset/ccpd_2019.yaml"
PROJECT_PATH = '/leonardo/home/userexternal/gdaddari/Computer-Vision/src/YOLO/runs'
YOLO_MODEL_PATH = 'src/YOLO/runs/train/weights/best.pt'

# -------- Dataset Globals --------- #
IMG_WIDTH = 720
IMG_HEIGHT = 1160
CLASS_ID = 0 

PROVINCES = ["皖", "沪", "津", "渝", "冀", "晋", "蒙", "辽", "吉", "黑", "苏", "浙", "京", "闽", "赣", "鲁", "豫", "鄂", "湘", "粤", "桂", "琼", "川", "贵", "云", "藏", "陕", "甘", "青", "宁", "新", "警", "学", "O"]
ALPHABETS = ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'J', 'K', 'L', 'M', 'N', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'O']
ADS = ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'J', 'K', 'L', 'M', 'N', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', 'O']

charset = PROVINCES + [c for c in ALPHABETS if c not in PROVINCES] + [str(i) for i in range(10)]
charset = list(dict.fromkeys(charset)) 

# -------- BASELINE specific globals --------- #
W_RESIZE = 224
H_RESIZE =224
X_SCALE = W_RESIZE/IMG_WIDTH
Y_SCALE = H_RESIZE/IMG_HEIGHT

transform_detection = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor()
])

transform_recognition = transforms.Compose([
    transforms.Resize((64, 128)),  
    transforms.ToTensor(),
])

# -------- GPU support --------- #
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Utils

In [ ]:
def compute_iou(preds, gts):
   
    intersection_x1 = np.maximum(preds[:, 0], gts[:, 0])
    intersection_y1 = np.maximum(preds[:, 1], gts[:, 1])
    intersection_x2 = np.minimum(preds[:, 2], gts[:, 2])
    intersection_y2 = np.minimum(preds[:, 3], gts[:, 3])

    intersection_w = np.maximum(0,  intersection_x2 - intersection_x1)
    intersection_h = np.maximum(0,  intersection_y2 - intersection_y1)
    intersection_area = intersection_w * intersection_h

    area_preds = (preds[:, 2] - preds[:, 0]) * (preds[:, 3] - preds[:, 1])
    area_gts = (gts[:, 2] - gts[:, 0]) * (gts[:, 3] - gts[:, 1])

    union_area = area_preds + area_gts - intersection_area

    iou = intersection_area / (union_area + 1e-7) 

    return iou

def plot_epoch_losses(epoch_losses, title="Training Loss per Epoch"):
    plt.figure(figsize=(8, 5))
    plt.plot(range(1, len(epoch_losses)+1), epoch_losses, marker='o', color='blue')
    plt.title(title)
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.grid(True)
    plt.xticks(range(1, len(epoch_losses)+1))
    plt.show()

def draw_bbox(img_path, label_path, out_path):
    """
    Plots and saves an example of boxplot extracted from the filename and superposed to the image.
    """
    with open(label_path) as f:
        _, xc, yc, w, h = map(float, f.readline().split())

    img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    xc, yc, w, h = xc * IMG_WIDTH, yc * IMG_HEIGHT, w * IMG_WIDTH, h * IMG_HEIGHT
    x1, y1 = xc - w/2, yc - h/2

    fig, ax = plt.subplots()
    ax.imshow(img)
    ax.add_patch(patches.Rectangle((x1, y1), w, h, edgecolor='r', facecolor='none', linewidth=2))
    ax.axis('off')
    out_path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(out_path, bbox_inches='tight', pad_inches=0)
    plt.show()
    plt.close()
    print(f"Image saved in: {out_path}")

# Data

## Dataset download and setup

In [ ]:
SO="MacOs"
# Installing pixz for faster unxipping
if SO=="Linux":
    !apt-get update
    !apt-get install pixz
elif SO=="MacOs":
    !brew install pixz

In [ ]:
# Downloading the .tar dataset and extracting it
%cd dataset
!gdown --id 1HDyFIuH65kVLtsXqxLA8gs0gJr7CRynh
!tar -I 'pixz -d' -xf CCPD2019.tar.xz

In [ ]:
######################## CONVERTING THE DATASET IN YOLO FORMAT ########################

def convert_bbox(x1, y1, x2, y2):
    """
    Converts bbox coordinates in pixels (normalized), following the YOLO format.
    """
    bbox_width = abs(x2 - x1)
    bbox_height = abs(y2 - y1)
    x_center = x1 + bbox_width / 2.0
    y_center = y1 + bbox_height / 2.0

    # Normalizing
    x_center /= IMG_WIDTH
    y_center /= IMG_HEIGHT
    bbox_width /= IMG_WIDTH
    bbox_height /= IMG_HEIGHT

    return x_center, y_center, bbox_width, bbox_height

def parse_filename(fname):
    """
    Extracts bbox coordinates from the image file name and converts them in YOLO format
    """
    fname = Path(fname)
    parts = fname.stem.split('-')
    if len(parts) != 7:
        return None

    try:
        bbox_str = parts[2]
        x1y1_str, x2y2_str = bbox_str.split('_')
        x1, y1 = map(int, x1y1_str.split('&'))
        x2, y2 = map(int, x2y2_str.split('&'))

        return convert_bbox(x1, y1, x2, y2)
    except Exception as e:
        print(f"[WARN] Could not parse bbox from file '{fname}': {e}")
        return None

def process_single_image(args):
    img_path, images_dest, labels_dest = args
    bbox = parse_filename(img_path.name)
    if bbox is None:
        return

    shutil.copy(img_path, images_dest / img_path.name)
    label_path = labels_dest / (img_path.stem + ".txt")
    with open(label_path, 'w') as f:
        f.write(f"{CLASS_ID} {' '.join(f'{x:.6f}' for x in bbox)}\n")


def process_images(images, images_src, dest_root, split, leonardo=False):
    images_dest = Path(dest_root) / "images" / split
    labels_dest = Path(dest_root) / "labels" / split
    os.makedirs(images_dest, exist_ok=True)
    os.makedirs(labels_dest, exist_ok=True)

    args_list = [(img_path, images_dest, labels_dest) for img_path in images]

    if leonardo:
        with ProcessPoolExecutor(max_workers=multiprocessing.cpu_count()) as executor:
            list(tqdm(executor.map(process_single_image, args_list), total=len(args_list), desc=f"Processing {split} set"))
    else:
        for args in tqdm(args_list, desc=f"Processing {split} set"):
            process_single_image(args)

    print(f"{split} set saved to {images_dest} and {labels_dest}")


def prepare_ccpd_base(src_root, dest_root="CCPD_YOLO", split_ratio=0.8, seed=42):
    """
    Builds the training subdataset 'ccpd_base'.
    """
    src = "ccpd_base"
    images_src = Path(os.path.join(src_root, src))
    image_files = list(images_src.glob("*.jpg"))
    print(f"[INFO] Found {len(image_files)} images in {images_src}")
    random.seed(seed)
    random.shuffle(image_files)

    split_index = int(len(image_files) * split_ratio)
    train_files = image_files[:split_index]
    val_files = image_files[split_index:]

    process_images(train_files, images_src, f"{dest_root}/{src}", "train")
    process_images(val_files, images_src, f"{dest_root}/{src}", "val")

def prepare_other_subset(src_root, subset, dest_root="CCPD_YOLO"):
    """
    Builds the other subdatasets for the testing phase (individually).
    """
    images_src = Path(os.path.join(src_root, subset))
    image_files = list(images_src.glob("*.jpg"))
    process_images(image_files, images_src, f"{dest_root}/{subset}", "test")


##################################################################

In [ ]:
## Running the preprocessing in order to build the dataset in YOLO format

print("Starting preprocessing...")
#scratch_dir = os.environ.get("SCRATCH", "/leonardo_scratch/large/userexternal/gdaddari")
#SRC_ROOT = os.path.join(scratch_dir, "dataset", "CCPD2019")
#base_dest = os.path.join(scratch_dir, "dataset", "CCPD_YOLO")

SRC_ROOT = os.path.join("dataset", "CCPD2019")
base_dest = os.path.join("dataset", "CCPD_YOLO")

other_subsets = [
    "ccpd_blur", "ccpd_challenge", "ccpd_db",
    "ccpd_fn", "ccpd_rotate", "ccpd_tilt", "ccpd_weather"
]

print("Training subset...")
prepare_ccpd_base(SRC_ROOT, dest_root=base_dest)

print("Other subsets...")
for subset in other_subsets:
    print(f"Processing {subset}...")
    prepare_other_subset(SRC_ROOT, subset, base_dest)

In [ ]:
img_dir = Path("dataset/CCPD_YOLO/ccpd_base/images/train")
lbl_dir = Path("dataset/CCPD_YOLO/ccpd_base/labels/train")
out_dir = Path("dataset/CCPD_YOLO")

img = sorted(img_dir.glob("*.jpg"))[0]
lbl = lbl_dir / f"{img.stem}.txt"
out = out_dir / f"bbox-example_{img.name}"
draw_bbox(img, lbl, out)

## Class Definitions

In [ ]:
class CCPDImage:
    def __init__(self, filename):
        self.filename = Path(filename)
        self.valid = self._parse()

    def _parse(self):
        parts = self.filename.stem.split('-')
        if len(parts) != 7:
            return False
        self.parts = parts
        return True

    @property
    def plate_code(self):
        try:
            code = list(map(int, self.parts[4].split('_')))
            return code
        except Exception:
            return None

    @property
    def plate_str(self):
        try:
            code = self.plate_code
            province = PROVINCES[code[0]]
            letter = ALPHABETS[code[1]]
            tail = ''.join(ADS[i] for i in code[2:])
            return province + letter + tail
        except Exception:
            return "INVALID"

    @property
    def bbox_absolute(self):
        try:
            bbox_str = self.parts[2]
            x1y1_str, x2y2_str = bbox_str.split('_')
            x1, y1 = map(int, x1y1_str.split('&'))
            x2, y2 = map(int, x2y2_str.split('&'))
            return torch.tensor([x1, y1, x2, y2], dtype=torch.float)
        except Exception:
            return None

    @property
    def bbox_yolo(self):
        try:
            x1, y1, x2, y2 = self.bbox_absolute
            bbox_width = abs(x2 - x1)
            bbox_height = abs(y2 - y1)
            x_center = x1 + bbox_width / 2.0
            y_center = y1 + bbox_height / 2.0

            x_center /= IMG_WIDTH
            y_center /= IMG_HEIGHT
            bbox_width /= IMG_WIDTH
            bbox_height /= IMG_HEIGHT

            return (x_center, y_center, bbox_width, bbox_height)
        except Exception:
            return None

    def __repr__(self):
        return f"CCPDImageInfo(plate='{self.plate_str}', valid={self.valid})"
    
class CCPDDataset(Dataset):
    def __init__(self, img_dir, transform=None, task="detection"):
        """
        task: 'detection' | 'recognition'
        """
        self.img_dir = Path(img_dir)
        self.task = task

        # If not transform and we're in recognition task, use FullRobustAugmentation
        if transform is None and task == "recognition":
            self.transform = FullRobustAugmentation()
        else:
            self.transform = transform

        self.image_paths = [p for p in self.img_dir.glob("*.jpg")]
        self.image_objs = [CCPDImage(p) for p in self.image_paths if CCPDImage(p).valid]

    def __len__(self):
        return len(self.image_objs)

    def __getitem__(self, idx):
        img_obj = self.image_objs[idx]
        img_path = img_obj.filename
        image = Image.open(img_path).convert("RGB")

        if self.task == "detection":
            # Image full + bbox
            if self.transform:
                image = self.transform(image)
            bbox = img_obj.bbox_absolute  # torch.tensor([x1, y1, x2, y2])
            return image, bbox

        elif self.task == "recognition":
            x1, y1, x2, y2 = img_obj.bbox_absolute.tolist()
            image = image.crop((x1, y1, x2, y2))

            if self.transform:
                image = self.transform(image)

            plate_code = img_obj.plate_code
            return image, torch.tensor(plate_code)

        else:
            raise ValueError(f"Task '{self.task}' not allowed.")
        
class FullRobustAugmentation:
    def __init__(self):
        self.base = transforms.Compose([
            transforms.Resize((48, 144)),
            transforms.ColorJitter(brightness=0.6, contrast=0.6, saturation=0.3, hue=0.1),
            transforms.RandomRotation(degrees=30),
            transforms.RandomAffine(degrees=0, shear=10),
            transforms.RandomPerspective(distortion_scale=0.4, p=0.5),
            transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),
        ])


       
    def __call__(self, img):
        img = self.base(img)  # geometrie e jitter

        if random.random() < 0.5:
            img = self.random_motion_blur(img)

        if random.random() < 0.5:
            factor = random.uniform(0.3, 1.8)
            img = TF.adjust_brightness(img, factor)

        if random.random() < 0.5:
            img = self.random_occlusion(img)

        if random.random() < 0.5:
            img = self.random_compression(img)

        if random.random() < 0.5:
            img = self.add_fog(img)

        return TF.to_tensor(img)


    def random_motion_blur(self, img):
        kernel_size = random.choice([5, 9, 15])
        return img.filter(ImageFilter.GaussianBlur(radius=kernel_size / 5))

    def add_fog(self, img):
        fog = Image.new("RGB", img.size, color=(200, 200, 200))
        return Image.blend(img, fog, alpha=random.uniform(0.1, 0.4))


    def random_occlusion(self, img):
        draw = img.copy()
        w, h = draw.size
        x0 = random.randint(0, w // 2)
        y0 = random.randint(0, h // 2)
        x1 = x0 + random.randint(10, 40)
        y1 = y0 + random.randint(10, 20)
        color = random.choice([(0, 0, 0), (255, 255, 255)])
        for x in range(x0, min(x1, w)):
            for y in range(y0, min(y1, h)):
                draw.putpixel((x, y), color)
        return draw

    def random_compression(self, img):
        buffer = io.BytesIO()
        quality = random.randint(10, 40)
        img.save(buffer, format="JPEG", quality=quality)
        buffer.seek(0)
        return Image.open(buffer)


In [ ]:
# Loading datasets
train_dataset_det = CCPDDataset(TRAINING_PATH, transform=transform_detection, task='detection')
train_loader_det = DataLoader(train_dataset_det, batch_size=32, shuffle=True)

train_dataset_rec = CCPDDataset(TRAINING_PATH, transform=transform_recognition, task='recognition')
train_loader_rec = DataLoader(train_dataset_rec, batch_size=32, shuffle=True)
train_dataset_rec_pdlpr = CCPDDataset(TRAINING_PATH, task='recognition')
train_loader_rec_pdlpr = DataLoader(train_dataset_rec_pdlpr, batch_size=32, shuffle=True)

test_dataset_det = CCPDDataset(TEST_PATH, transform=transform_detection, task='detection')
test_loader_det = DataLoader(test_dataset_det, batch_size=32, shuffle=True)
test_dataset_rec = CCPDDataset(TEST_PATH, transform=transform_recognition, task='recognition')
test_loader_rec = DataLoader(test_dataset_rec, batch_size=32, shuffle=True)

# Network

## Baseline

In [ ]:
class BaselineModel(nn.Module):
    def __init__(self, num_classes_list, pretrained=True):
        super().__init__()

        # Detection backbone
        resnet_det = models.resnet34(pretrained=pretrained)
        self.backbone_det = nn.Sequential(*list(resnet_det.children())[:-2])
        self.pool_det = nn.AdaptiveAvgPool2d((1, 1))
        self.regressor = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 4),
            nn.Sigmoid()
        )

        # Recognition backbone
        resnet_rec = models.resnet34(pretrained=pretrained)
        self.backbone_rec = nn.Sequential(*list(resnet_rec.children())[:-2])
        self.pool_rec = nn.AdaptiveAvgPool2d((1, 1))
        self.classifiers = nn.ModuleList([
            nn.Linear(512, n_classes) for n_classes in num_classes_list
        ])

    def forward_backbone_det(self, x):
        x = self.backbone_det(x)
        x = self.pool_det(x)
        return x.view(x.size(0), -1)

    def forward_backbone_rec(self, x):
        x = self.backbone_rec(x)
        x = self.pool_rec(x)
        return x.view(x.size(0), -1)

    def forward_detection(self, x):
        x = self.forward_backbone_det(x)
        return self.regressor(x)

    def forward_recognition(self, x):
        x = self.forward_backbone_rec(x)
        return [clf(x) for clf in self.classifiers]

    def forward(self, x, mode=None):
        """
        mode: 'detection' or 'recognition'
        """
        if mode == 'detection':
            return self.forward_detection(x)
        elif mode == 'recognition':
            return self.forward_recognition(x)
        else:
            raise ValueError(f"Unsupported mode: {mode}")

## YOLOv5-PDLPR

In [ ]:
class YOLOv5: #Wrapper
    def __init__(self, model_path):
        self.model_path = model_path
        self.model_type = "yolov5"

### PDLRP

In [ ]:
def collate_fn(batch):
    images, texts = zip(*batch)
    images = torch.stack(images)
    token_seqs = [torch.tensor(tokenizer.encode(t)[:seq_len] + [0]*(seq_len-len(t))) for t in texts]
    targets = torch.stack(token_seqs)  # [B, seq_len]
    if (targets >= num_classes).any() or (targets < 0).any():
        print("[ERROR] Out of range, Excerpt:")
        for t in texts:
            print("Label:", t, "Encoded:", tokenizer.encode(t))
        print("Target tensor:", targets)
        print("num_classes:", num_classes)
        raise ValueError("Out of range For CrossEntropyLoss!")
    return images, targets

class SimplePlateTokenizer:
    def __init__(self, charset):
        self.char2idx = {c: i + 1 for i, c in enumerate(charset)}  # 0 = PAD
        self.char2idx['<PAD>'] = 0
        self.idx2char = {i: c for c, i in self.char2idx.items()}
    def encode(self, text):
        for c in text:
            if c not in self.char2idx:
                print(f"[Tokenizer Warning] Carattere '{c}' non nel charset! Verrà codificato come PAD (0)")
        return [self.char2idx.get(c, 0) for c in text]
    def decode(self, indices):
        return ''.join([self.idx2char.get(i, '') for i in indices if i != 0])
    def vocab_size(self):
        return len(self.char2idx)

tokenizer = SimplePlateTokenizer(charset)
num_classes = tokenizer.vocab_size()
seq_len = 8  # maximum car plate length

# --- IGFE ---
class FocusStructure(nn.Module):
    def __init__(self):
        super().__init__()
    def forward(self, x):
        return torch.cat([
            x[..., ::2, ::2],
            x[..., 1::2, ::2],
            x[..., ::2, 1::2],
            x[..., 1::2, 1::2]
        ], dim=1)

class CNNBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, padding=1):
        super().__init__()
        self.leaky_relu = nn.LeakyReLU(0.2, inplace=False)
        self.bn = nn.BatchNorm2d(in_channels)
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding)
    def forward(self, x):
        x = self.leaky_relu(x)
        x = self.bn(x)
        x = self.conv(x)
        return x

class ResBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.cnn_block1 = CNNBlock(in_channels, out_channels)
        self.cnn_block2 = CNNBlock(out_channels, out_channels)
        self.identity = nn.Identity() if in_channels == out_channels else nn.Conv2d(in_channels, out_channels, 1)
    def forward(self, x):
        identity = self.identity(x)
        out = self.cnn_block1(x)
        out = self.cnn_block2(out)
        return out + identity

class ConvDownSampling(nn.Module):
    def __init__(self, in_channels, out_channels, stride=2):
        super().__init__()
        self.leaky_relu = nn.LeakyReLU(0.2, inplace=False)
        self.bn = nn.BatchNorm2d(in_channels)
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1)
    def forward(self, x):
        x = self.leaky_relu(x)
        x = self.bn(x)
        x = self.conv(x)
        return x

class IGFE(nn.Module):
    def __init__(self, in_channels, base_channels):
        super().__init__()
        self.focus = FocusStructure()
        self.layer1 = ResBlock(4 * in_channels, base_channels)
        self.layer2 = ResBlock(base_channels, base_channels)
        self.down1 = ConvDownSampling(base_channels, base_channels, stride=2)
        self.layer3 = ResBlock(base_channels, base_channels)
        self.layer4 = ResBlock(base_channels, base_channels)
        self.down2 = ConvDownSampling(base_channels, base_channels, stride=2)
    def forward(self, x):
        x = self.focus(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.down1(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.down2(x)
        return x
    
# --- Encoder ---
class PositionalEncoding2D(nn.Module):
    def __init__(self, d_model, height, width):
        super().__init__()
        if d_model % 4 != 0:
            raise ValueError("d_model must be divisible by 4 for 2D positional encoding")
        pe = torch.zeros(d_model, height, width)
        y_pos = torch.arange(0, height).unsqueeze(1).repeat(1, width)
        x_pos = torch.arange(0, width).unsqueeze(0).repeat(height, 1)
        div_term = torch.exp(torch.arange(0, d_model // 2, 2) * -(torch.log(torch.tensor(10000.0)) / (d_model // 2)))
        pe[0::4, :, :] = torch.sin(y_pos.unsqueeze(0) * div_term.unsqueeze(1).unsqueeze(2))
        pe[1::4, :, :] = torch.cos(y_pos.unsqueeze(0) * div_term.unsqueeze(1).unsqueeze(2))
        pe[2::4, :, :] = torch.sin(x_pos.unsqueeze(0) * div_term.unsqueeze(1).unsqueeze(2))
        pe[3::4, :, :] = torch.cos(x_pos.unsqueeze(0) * div_term.unsqueeze(1).unsqueeze(2))
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x):
        return x + self.pe[:, :x.size(1), :x.size(2), :x.size(3)]

class EncoderModule(nn.Module):
    def __init__(self, d_model=512, nhead=8, height=16, width=16):
        super().__init__()
        self.pos_enc = PositionalEncoding2D(d_model, height, width)
        self.cnn_block1 = CNNBlock(d_model, d_model, kernel_size=1, stride=1, padding=0)
        self.mha = nn.MultiheadAttention(embed_dim=d_model, num_heads=nhead)
        self.cnn_block2 = CNNBlock(d_model, d_model, kernel_size=1, stride=1, padding=0)
        self.add_norm = nn.LayerNorm(d_model)
    def forward(self, x):
        residual = x.clone()
        x = self.pos_enc(x)
        x = self.cnn_block1(x)
        B, C, H, W = x.shape
        x_ = x.permute(2, 3, 0, 1).reshape(H*W, B, C)
        attn_out, _ = self.mha(x_, x_, x_)
        x = attn_out.reshape(H, W, B, C).permute(2, 3, 0, 1)
        x = self.cnn_block2(x)
        out = residual + x
        out = out.permute(0, 2, 3, 1)
        out = self.add_norm(out)
        out = out.permute(0, 3, 1, 2)
        return out

class Encoder(nn.Module):
    def __init__(self, d_model=512, nhead=8, height=16, width=16, num_layers=3):
        super().__init__()
        self.layers = nn.ModuleList([
            EncoderModule(d_model, nhead, height, width) for _ in range(num_layers)
        ])
    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

# --- Decoder ---
class AddNorm(nn.Module):
    def __init__(self, d_model, eps=1e-6):
        super().__init__()
        self.norm = nn.LayerNorm(d_model, eps=eps)
    def forward(self, x, sublayer_out):
        return self.norm(x + sublayer_out)

class DecodingModule(nn.Module):
    def __init__(self, d_model=512, nhead=8, height=16, width=16):
        super().__init__()
        self.pos_enc = PositionalEncoding2D(d_model, height, width)
        self.self_attn = nn.MultiheadAttention(embed_dim=d_model, num_heads=nhead)
        self.cross_cnn1 = CNNBlock(d_model, d_model, kernel_size=1, stride=1, padding=0)
        self.cross_cnn2 = CNNBlock(d_model, d_model, kernel_size=1, stride=1, padding=0)
        self.cross_attn = nn.MultiheadAttention(embed_dim=d_model, num_heads=nhead)
        self.feed_forward = nn.Sequential(
            nn.Conv2d(d_model, d_model * 4, kernel_size=1),
            nn.ReLU(inplace=False),
            nn.Conv2d(d_model * 4, d_model, kernel_size=1),
        )
        self.addnorm1 = AddNorm(d_model)
        self.addnorm2 = AddNorm(d_model)
        self.addnorm3 = AddNorm(d_model)
    def forward(self, x, encoder_out):
        x = self.pos_enc(x)
        B, C, H, W = x.shape
        x_ = x.permute(2, 3, 0, 1).reshape(H*W, B, C)
        self_attn_out, _ = self.self_attn(x_, x_, x_)
        self_attn_out = self.addnorm1(x_.permute(1, 0, 2), self_attn_out.permute(1, 0, 2))
        self_attn_out = self_attn_out.permute(1, 0, 2)
        x = self_attn_out.reshape(H, W, B, C).permute(2, 3, 0, 1)
        enc = self.cross_cnn1(encoder_out)
        enc = self.cross_cnn2(enc)
        B_enc, C_enc, H_enc, W_enc = enc.shape
        enc_ = enc.permute(2, 3, 0, 1).reshape(H_enc*W_enc, B_enc, C_enc)
        x_ = x.permute(2, 3, 0, 1).reshape(H*W, B, C)
        cross_attn_out, _ = self.cross_attn(x_, enc_, enc_)
        cross_attn_out = self.addnorm2(x_.permute(1, 0, 2), cross_attn_out.permute(1, 0, 2))
        cross_attn_out = cross_attn_out.permute(1, 0, 2)
        x = cross_attn_out.reshape(H, W, B, C).permute(2, 3, 0, 1)
        ff_out = self.feed_forward(x)
        out = self.addnorm3(x.permute(0, 2, 3, 1).reshape(B, -1, C), ff_out.permute(0, 2, 3, 1).reshape(B, -1, C))
        out = out.reshape(B, H, W, C).permute(0, 3, 1, 2)
        return out

class Decoder(nn.Module):
    def __init__(self, d_model=512, nhead=8, height=16, width=16, num_layers=3, num_classes=68, seq_len=8):
        super().__init__()
        self.layers = nn.ModuleList([
            DecodingModule(d_model=d_model, nhead=nhead, height=height, width=width)
            for _ in range(num_layers)
        ])
        self.seq_len = seq_len
        self.classifier = nn.Linear(d_model, num_classes)
        self.pool = nn.AdaptiveAvgPool2d((1, seq_len))  # (B, C, 1, seq_len)
    def forward(self, x, encoder_out):
        for layer in self.layers:
            x = layer(x, encoder_out)
        x = self.pool(x)  # (B, C, 1, seq_len)
        x = x.squeeze(2)  # (B, C, seq_len)
        x = x.permute(0, 2, 1)  # (B, seq_len, C)
        logits = self.classifier(x)  # (B, seq_len, num_classes)
        return logits

## ----- PDLPR MODEL ----- ##
class PDLPR(nn.Module):
    def __init__(self,
                 in_channels=3,
                 base_channels=512,
                 encoder_d_model=512,
                 encoder_nhead=8,
                 encoder_height=16,
                 encoder_width=16,
                 decoder_num_layers=3,
                 num_classes=68,
                 seq_len=8):
        super().__init__()
        
        self.model_type = "pdlpr"
        
        self.igfe = IGFE(in_channels, base_channels)
        
        self.pool = nn.AdaptiveAvgPool2d((encoder_height, encoder_width))
        
        self.encoder = Encoder(
            d_model=encoder_d_model, 
            nhead=encoder_nhead, 
            height=encoder_height, 
            width=encoder_width
        )
        
        self.decoder = Decoder(
            d_model=encoder_d_model,
            nhead=encoder_nhead,
            height=encoder_height,
            width=encoder_width,
            num_layers=decoder_num_layers,
            num_classes=num_classes,
            seq_len=seq_len
        )
        
    def forward(self, x):
        x = self.igfe(x)
        x = self.pool(x)
        x = self.encoder(x)
        decoder_input = torch.zeros_like(x)
        x = self.decoder(decoder_input, x)
        return x
    

### Complete pipeline

In [ ]:
class YOLOv5PDLPR_Pipeline:
    def __init__(self, yolo_weights_path, pdlpr_model, tokenizer, device='cuda'):
        self.yolo = torch.hub.load('ultralytics/yolov5', 'custom', path=yolo_weights_path).to(device).eval()
        
        self.pdlpr = pdlpr_model.to(device).eval()
        self.tokenizer = tokenizer
        self.device = device

        self.transform = transforms.Compose([
            transforms.Resize((64, 128)), #TODO: check
            transforms.ToTensor(),
        ])

    def detect_plates(self, image_pil):
        results = self.yolo(image_pil)
        return results.xyxy[0]

    def recognize_plate(self, crop):
        crop_tensor = self.transform(crop).unsqueeze(0).to(self.device)
        with torch.no_grad():
            logits = self.pdlpr(crop_tensor)
            pred = torch.argmax(logits, dim=-1).squeeze(0)
            plate = self.tokenizer.decode(pred.tolist())
        return plate

    def __call__(self, image_pil):
        detections = self.detect_plates(image_pil)
        plates = []
        for det in detections:
            x1, y1, x2, y2, conf, cls = det.tolist()
            crop = image_pil.crop((int(x1), int(y1), int(x2), int(y2)))
            plate_text = self.recognize_plate(crop)
            plates.append({
                "text": plate_text,
                "bbox": (x1, y1, x2, y2),
                "confidence": conf
            })
        return plates

# Train

In [ ]:
class Trainer:
    def __init__(self, model, task, device, lr=1e-3, num_classes_list=None):
        self.model = model.to(device)
        self.task = task
        self.device = device
        self.optimizer = optim.Adam(self.model.parameters(), lr=lr)
        self.losses = []

        # Recognizing model
        self.model_type = model.model_type if hasattr(model, "model_type") else "baseline"
        self.model_path = model.model_path if hasattr(model, "model_path") else "" 

        self.set_task(task, num_classes_list)

    def plot_epoch_losses(self, losses, title, save_dir):
        os.makedirs(save_dir, exist_ok=True)
        plt.figure(figsize=(10, 5))
        plt.plot(range(1, len(losses)+1), losses, label='Loss', marker='o')
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.title(title)
        plt.grid(True)
        plt.legend()
        plt.tight_layout()
        save_path = os.path.join(save_dir, f"{title.replace(' ', '_').lower()}.png")
        plt.savefig(save_path)
        plt.close()
        print(f"Loss graph saved in: '{save_path}'")

    def set_task(self, task, num_classes_list=None):
        self.task = task
        if task == "detection":
            self.criterion = nn.MSELoss()
            self.criterions = None  # Reset
        elif task == "recognition":
            if num_classes_list is None:
                raise ValueError("num_classes_list needed.")
            self.criterions = [nn.CrossEntropyLoss() for _ in num_classes_list]
            self.criterion = None  # Reset
        else:
            raise ValueError(f"Task not allowed: {task}")
    
    def train(self, dataloader=None, epochs=10, batch_size=50, optimizer="Adam",lr0=1e-3,lrf=1e-5, cos_lr=True,project="runs/train", name="lp_detection", cache="ram"):
        if self.model_type == "yolov5":
            return self._train_yolov5(epochs=epochs, batch_size=batch_size, optimizer=optimizer, lr0=lr0, lrf=lrf, cos_lr=cos_lr, project=project, name=name, cache=cache)
        elif self.model_type == "pdlpr":
            return self._train_pdlpr(dataloader, epochs)
        else:
            return self._train_baseline(dataloader, epochs)

    def _train_baseline(self, dataloader, epochs):
        self.model.train()
        for epoch in range(epochs):
            total_loss = 0
            for images, targets in tqdm(dataloader, desc=f"[{self.task}] Epoch {epoch+1}/{epochs}"):
                images = images.to(self.device)
                if self.task == "detection":
                    bboxes = targets.to(self.device)
                    norm = torch.tensor([IMG_WIDTH, IMG_HEIGHT, IMG_WIDTH, IMG_HEIGHT]).to(self.device)
                    bboxes = bboxes / norm
                    preds = self.model(images, 'detection')
                    loss = self.criterion(preds, bboxes)
                elif self.task == "recognition":
                    labels = targets.to(self.device)
                    outputs = self.model(images, 'recognition')
                    loss = 0
                    for i, crit in enumerate(self.criterions):
                        loss += crit(outputs[i], labels[:, i])
                self.optimizer.zero_grad()
                loss.backward()
                self.optimizer.step()
                total_loss += loss.item()
            avg_loss = total_loss / len(dataloader)
            print(f"Epoch {epoch+1} - Loss: {avg_loss:.4f}")
            self.losses.append(avg_loss)
        self.plot_epoch_losses(self.losses, f"{self.task.capitalize()} Loss", "results")
        return self.model

    def _train_yolov5(self, epochs=25, batch_size=50, optimizer="Adam", lr0=1e-3, lrf=1e-5, cos_lr=True, project="runs/train", name="lp_detection", cache="ram"):
        YOLOv5_training(
            weights=self.model_path,
            data=DATASET_PATH_YOLO,
            epochs=epochs,
            batch_size=batch_size,
            imgsz=640,
            optimizer=optimizer,
            lr0=lr0,
            lrf=lrf,
            cos_lr=cos_lr,
            project=project,
            name=name,
            cache=cache
        )
        return self.model

    def _train_pdlpr(self, dataloader, epochs):
        self.model.train()
        loss_fn = nn.CrossEntropyLoss(ignore_index=0)
        scaler = GradScaler(device="cuda" if torch.cuda.is_available() else "cpu")

        train_losses = []

        for epoch in range(epochs):
            running_loss = 0.0
            pbar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{epochs} [PDLPR Train]", unit="batch")
            for images, targets in pbar:
                images = images.to(self.device)
                targets = targets.to(self.device)

                self.optimizer.zero_grad()
                with autocast(device_type="cuda"):
                    output = self.model(images)
                    output = output.permute(0, 2, 1)  # [B, SeqLen, C] -> [B, C, SeqLen]
                    loss = loss_fn(output, targets)

                scaler.scale(loss).backward()
                scaler.step(self.optimizer)
                scaler.update()

                running_loss += loss.item()
                pbar.set_postfix({"batch_loss": loss.item()})

            avg_loss = running_loss / len(dataloader)
            train_losses.append(avg_loss)
            print(f"Epoch [{epoch+1}/{epochs}] - Train Loss: {avg_loss:.4f}")

            torch.save(self.model.state_dict(), f"models/PDLPR/weights/pdlpr_epoch{epoch+1}.pth")

        # Save final model
        torch.save(self.model.state_dict(), "models/PDLPR/weights/pdlpr_final.pth")

        # Plotting
        self.plot_epoch_losses(train_losses, "PDLPR Loss", "models/PDLPR/logs")
        return self.model

### Baseline

In [ ]:
num_classes_list = [len(PROVINCES), len(ALPHABETS)] + [len(ADS)] * 5 


# -------------------------
# Training detection model
# -------------------------
det_model = BaselineModel(num_classes_list=num_classes_list)
det_model.load_state_dict(torch.load("models/baseline/resnet34_detection_best.pth"))
det_trainer = Trainer(det_model, task="detection", device=DEVICE)
det_model = det_trainer.train(train_loader_det, epochs=20)


# -------------------------
# Training recognition model
# -------------------------
rec_model = BaselineModel(num_classes_list=num_classes_list)
rec_model.load_state_dict(torch.load("models/baseline/resnet34_recognition_best.pth"))
rec_trainer = Trainer(rec_model, task="recognition", device=DEVICE)
rec_model = rec_trainer.train(train_loader_rec, epochs=20)

### YOLOv5-PDLPR

In [ ]:
# -------------------------
# Training detection model (YOLOv5)
# -------------------------
yolov5_model = YOLOv5()
yolov5_trainer = Trainer(yolov5_model, task="detection", device=DEVICE)
yolov5_model = yolov5_trainer.train(epochs=25, 
                                    batch_size=50, 
                                    optimizer="Adam", 
                                    lr0=1e-3, 
                                    lrf=1e-5, 
                                    cos_lr=True, 
                                    project="runs/train", 
                                    name="lp_detection", 
                                    cache="ram"
                                    )

# -------------------------
# Training recognition model (PDLPR)
# -------------------------
pdlpr_model = PDLPR(
        in_channels=3,
        base_channels=256,
        encoder_d_model=256,
        encoder_nhead=4,
        encoder_height=16,
        encoder_width=16,
        decoder_num_layers=2,
        num_classes=num_classes,
        seq_len=seq_len
    )
pdlpr_trainer = Trainer(pdlpr_model, task="recognition", device=DEVICE)
pdlpr_model = pdlpr_trainer.train(train_loader_rec_pdlpr,epochs=25)


### PDLRP

In [ ]:
# ------ training ------ #


def PDLPR_training(image_folder,num_epochs, batch_size=32):

    # --- Training setup ---
    batch_size = 32
    dataset = CCPDPlateDataset(image_folder)

    # Split 80% and 20% for training and validation
    train_size = int(0.8 * len(dataset)) 
    val_size = len(dataset) - train_size
    train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn, num_workers=4)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn, num_workers=4)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = PDLPR(
        in_channels=3,
        base_channels=256,
        encoder_d_model=256,
        encoder_nhead=4,
        encoder_height=16,
        encoder_width=16,
        decoder_num_layers=2,
        num_classes=num_classes,
        seq_len=seq_len
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    loss_fn = nn.CrossEntropyLoss(ignore_index=0)
    scaler = GradScaler(device = "cuda")

    

    try:
        from tqdm import tqdm
    except ImportError:
        import subprocess
        subprocess.check_call(["pip", "install", "tqdm"])
        from tqdm import tqdm

    train_losses = []
    val_losses = []





        
        # --- Validation loop ---
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for images, targets in tqdm(val_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Val]", unit="batch"):
                images = images.to(device)
                targets = targets.to(device)
                with autocast(device_type="cuda"):
                    output = model(images)
                    output = output.permute(0, 2, 1)
                    loss = loss_fn(output, targets)
                val_loss += loss.item()
        avg_val_loss = val_loss / len(val_loader)
        val_losses.append(avg_val_loss)
        print(f"Epoch [{epoch+1}/{num_epochs}] - Val Loss: {avg_val_loss:.4f}")
        

        torch.save(model.state_dict(), f"src/PDLPR/weights/pdlpr_epoch{epoch+1}.pth")
    



    torch.save(model.state_dict(), "src/PDLPR/weights/pdlpr_final.pth")


    # --- Loss plot ---
    plt.figure(figsize=(10, 5))
    plt.plot(range(1, num_epochs+1), train_losses, label='Training Loss', marker='o')
    plt.plot(range(1, num_epochs+1), val_losses, label='Validation Loss', marker='x')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Training and Validation Loss')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig("src/PDLPR/logs/loss_plot.png")
    plt.close()
    print("Salvato grafico delle loss in 'src/PDLPR/loss_plot.png'")



if __name__ == "__main__":
    print("PDLPR Training ...")
    PDLPR_training(TRAINING_PATH_PDLPR, batch_size=32, num_epochs=3)

PDLPR Training ...


C:\Users\Lorenzo\AppData\Local\Temp\ipykernel_16484\1594025806.py:34: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
Epoch 1/3 [Train]:   0%|          | 0/5000 [00:00<?, ?batch/s]

# Evaluation

In [ ]:
class Metrics:
    def __init__(self, task):
        self.task = task
        self.reset()

    def reset(self):
        self.total_iou = []
        self.correct_chars = 0
        self.total_chars = 0
        self.correct_chars_wo_first = 0
        self.total_time = 0
        self.total_samples = 0
    
    @staticmethod
    def compute_iou(preds, gts):
        intersection_x1 = np.maximum(preds[:, 0], gts[:, 0])
        intersection_y1 = np.maximum(preds[:, 1], gts[:, 1])
        intersection_x2 = np.minimum(preds[:, 2], gts[:, 2])
        intersection_y2 = np.minimum(preds[:, 3], gts[:, 3])
        intersection_w = np.maximum(0,  intersection_x2 - intersection_x1)
        intersection_h = np.maximum(0,  intersection_y2 - intersection_y1)
        intersection_area = intersection_w * intersection_h
        area_preds = (preds[:, 2] - preds[:, 0]) * (preds[:, 3] - preds[:, 1])
        area_gts = (gts[:, 2] - gts[:, 0]) * (gts[:, 3] - gts[:, 1])
        union_area = area_preds + area_gts - intersection_area
        iou = intersection_area / (union_area + 1e-7) 
        return iou

    def update_detection(self, preds_abs, targets):
        ious = self.compute_iou(preds_abs, targets)  #IoU for each pair
        for iou in ious:
            self.total_iou.append(iou)          

    def update_recognition(self, outputs, targets):
        batch_size = targets.size(0)
        for i, output in enumerate(outputs):
            pred_label = output.argmax(dim=1)
            correct = (pred_label == targets[:, i]).sum().item()
            self.correct_chars += correct

            if i > 0:  # Without chinese char
                self.correct_chars_wo_first += correct

        self.total_chars += batch_size * len(outputs)

    def update_time(self, start_time, end_time, batch_size):
        self.total_time += end_time - start_time
        self.total_samples += batch_size

    def compute(self):
        results = {}
        if self.task == "detection":
            results['IoU'] = np.mean(self.total_iou)
        elif self.task == "recognition":
            results['Accuracy'] = self.correct_chars / self.total_chars
            if self.total_chars > 0:
                results['Accuracy_wo_first'] = self.correct_chars_wo_first / (self.total_chars - self.total_samples)
        results['FPS'] = self.total_samples / self.total_time if self.total_time > 0 else 0
        return results
    
class Evaluator:
    def __init__(self, model, device):
        self.model = model.to(device)
        self.device = device
        self.model_type = model.model_type if hasattr(model, "model_type") else "baseline"

    @torch.no_grad()
    def evaluate(self, dataloader, task):
        self.model.eval()
        metrics = Metrics(task=task)

        for images, targets in tqdm(dataloader, desc=f"[Evaluating {task}]"):
            start_time = time.time()
            images = images.to(self.device)

            if task == "detection":
                targets = targets.to(self.device)
                preds = self.model(images, mode='detection')
                preds_abs = preds * torch.tensor([IMG_WIDTH, IMG_HEIGHT, IMG_WIDTH, IMG_HEIGHT], device=self.device)
                metrics.update_detection(preds_abs.cpu().numpy(), targets.cpu().numpy())

            elif task == "recognition":
                targets = targets.to(self.device)
                outputs = self.model(images, mode='recognition')
                metrics.update_recognition(outputs, targets)

            end_time = time.time()
            metrics.update_time(start_time, end_time, images.size(0))

        return metrics.compute()

### Baseline

In [ ]:
# -------------------------
# Evaluation detection
# -------------------------
evaluator = Evaluator(det_model, device=DEVICE)
metrics_det = evaluator.evaluate(test_loader_det, task="detection")
print("Detection Results:", metrics_det)

# -------------------------
# Evaluation recognition
# -------------------------
evaluator = Evaluator(rec_model, device=DEVICE)
metrics_rec = evaluator.evaluate(test_loader_rec, task="recognition")
print("Recognition Results:", metrics_rec)

### YOLOv5

In [ ]:
YOLOv5_inference(
    weights=YOLO_MODEL_PATH,
    source=TEST_PATH_YOLOV5,
    imgsz=640,
    device="cuda:0",
    project=PROJECT_PATH,
    name="test",
    exist_ok=True
)

### PDLRP

In [ ]:
# ------ inference ------ #
def PDLPR_inference(dataset_folder, batch_size=64):

    dataset = CCPDPlateDataset(dataset_folder)
    dataloader = DataLoader(dataset, batch_size, shuffle=False, collate_fn=collate_fn, num_workers=4)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = PDLPR(
        in_channels=3,
        base_channels=256,
        encoder_d_model=256,
        encoder_nhead=4,
        encoder_height=16,
        encoder_width=16,
        decoder_num_layers=2,
        num_classes=num_classes,
        seq_len=seq_len
    ).to(device)
    model.load_state_dict(torch.load("src\PDLPR\weights\pdlpr_final.pth", map_location=device))
    model.eval()

    def decode_plate_pred(seq):
        if 0 in seq:
            seq = seq[:seq.index(0)]
        if len(seq) > 0:
            seq = seq[:-1]  # Remove last character (padding)
        return tokenizer.decode(seq)

    def decode_plate_gold(seq):
        if 0 in seq:
            seq = seq[:seq.index(0)]
        return tokenizer.decode(seq)

    correct = 0
    total = 0

    with torch.no_grad():
        for images, targets in tqdm(dataloader, desc="Inferenza CCPD"):
            images = images.to(device)
            outputs = model(images)  # [B, seq_len, num_classes]
            preds = outputs.argmax(dim=-1).cpu()  # [B, seq_len]
            for pred_seq, target_seq in zip(preds, targets):
                pred_plate = decode_plate_pred(pred_seq.tolist())
                gt_plate = decode_plate_gold(target_seq.tolist())
                if pred_plate == gt_plate:
                    correct += 1
                total += 1

    accuracy = correct / total if total > 0 else 0
    print(f"Accuracy su {total} immagini: {accuracy:.4f}")
    return accuracy




print("PDLPR Inference ...")
PDLPR_inference(TEST_PATH_PDLPR, batch_size=64)